# Epic 3 — Find Local Digital Help
## Library Services Dataset: Cleaning & Combination
**FIT5120 — Industry Experience Studio**

---

### Purpose
Clean and combine library location datasets from VIC, QLD, TAS, WA, and SA into a single standardised CSV for use in the Epic 3 Local Help Finder feature.

### Datasets
| File | State(s) | Records |
|------|----------|---------|
| all_library_services_combined.csv | VIC, QLD | 433 |
| cleaned_Libraries_TAS.csv | TAS | 46 |
| WA_Libraries.csv | WA | 235 |
| SA_LibraryLocations.xlsx | SA | 160 |

### Output
`src/data/clean_data/cleaned_Libraries_ALL.csv`

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

---
## 2. Set File Paths

In [ ]:
# Navigate from src/notebooks up to src/
src_base_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
raw_path = os.path.join(src_base_path, 'data', 'raw_data')
clean_path = os.path.join(src_base_path, 'data', 'clean_data')
os.makedirs(clean_path, exist_ok=True)

print('src base path :', src_base_path)
print('Raw data path :', raw_path)
print('Clean data path:', clean_path)
print('\nFiles in raw_data:')
print(os.listdir(raw_path))

---
## 3. Load Raw Datasets

In [ ]:
# Load all four source datasets
df_vicqld = pd.read_csv(os.path.join(raw_path, 'all_library_services_combined.csv'))
df_tas    = pd.read_csv(os.path.join(raw_path, 'cleaned_Libraries_TAS.csv'))
df_wa     = pd.read_csv(os.path.join(raw_path, 'WA_Libraries.csv'))
df_sa     = pd.read_excel(os.path.join(raw_path, 'SA_LibraryLocations.xlsx'))

print(f'VIC/QLD : {df_vicqld.shape[0]} rows x {df_vicqld.shape[1]} cols')
print(f'TAS     : {df_tas.shape[0]} rows x {df_tas.shape[1]} cols')
print(f'WA      : {df_wa.shape[0]} rows x {df_wa.shape[1]} cols')
print(f'SA      : {df_sa.shape[0]} rows x {df_sa.shape[1]} cols')

---
## 4. Exploratory Data Analysis
Before cleaning, inspect each dataset for structure, nulls, and quality issues.

### 4.1 VIC/QLD Combined — Existing Dataset

In [ ]:
print('Columns:', list(df_vicqld.columns))
print('\nNull counts:')
null_pct = (df_vicqld.isnull().sum() / len(df_vicqld) * 100).round(1)
print(pd.DataFrame({'Nulls': df_vicqld.isnull().sum(), 'Null %': null_pct}).to_string())
print('\nState breakdown:')
print(df_vicqld['state'].value_counts().to_string())

In [ ]:
df_vicqld.head()

### 4.2 TAS — Already Cleaned

In [ ]:
print('Columns:', list(df_tas.columns))
print('\nNull counts:')
print(df_tas.isnull().sum().to_string())

### 4.3 WA Libraries

In [ ]:
print('Columns:', list(df_wa.columns))
print('\nNull counts:')
null_pct = (df_wa.isnull().sum() / len(df_wa) * 100).round(1)
print(pd.DataFrame({'Nulls': df_wa.isnull().sum(), 'Null %': null_pct}).to_string())

In [ ]:
df_wa.head()

### 4.4 SA Libraries

In [ ]:
print('Columns:', list(df_sa.columns))
print('\nNull counts:')
null_pct = (df_sa.isnull().sum() / len(df_sa) * 100).round(1)
print(pd.DataFrame({'Nulls': df_sa.isnull().sum(), 'Null %': null_pct}).to_string())

In [ ]:
df_sa.head()

---
## 5. Target Schema

All datasets will be standardised to the following columns before combining:

| Column | Description |
|--------|-------------|
| `name` | Library name |
| `address` | Street address |
| `suburb` | Suburb or town |
| `state` | State abbreviation (VIC, QLD, TAS, WA, SA) |
| `postcode` | 4-digit postcode as string |
| `latitude` | Decimal latitude |
| `longitude` | Decimal longitude |
| `phone` | Contact phone number |
| `opening_hours` | Opening hours (null where not available) |
| `venue_type` | Always 'Library' for this dataset |
| `source` | Source dataset identifier for data lineage |

---
## 6. Clean Each Dataset

### 6.1 VIC/QLD — Fix 263 Null Names
263 out of 433 records have no name in the source data.
We fill them using `suburb + ' Library'` as a readable fallback.

In [ ]:
vicqld = df_vicqld.copy()

# Fill null names with suburb + 'Library' as a readable fallback
null_name_count = vicqld['name'].isnull().sum()
vicqld['name'] = vicqld.apply(
    lambda row: f"{row['suburb']} Library" if pd.isnull(row['name']) else row['name'],
    axis=1
)

# Standardise postcode to string
vicqld['postcode'] = vicqld['postcode'].astype(str).str.strip().str.split('.').str[0]

# Add required schema columns
vicqld['venue_type'] = 'Library'
vicqld['source']     = vicqld['state'].apply(lambda s: f'Libraries_{s}')

# Strip whitespace from string columns
str_cols = vicqld.select_dtypes(include='object').columns
vicqld[str_cols] = vicqld[str_cols].apply(lambda col: col.str.strip())

print(f'Null names fixed : {null_name_count} → {vicqld["name"].isnull().sum()}')
print(f'Rows             : {len(vicqld)}')
print(f'State breakdown  :')
print(vicqld['state'].value_counts().to_string())

### 6.2 TAS — Already Standardised
TAS was cleaned in a previous step. We just drop the extra columns (`is_aged_care_hub`) not needed in the final schema.

In [ ]:
tas = df_tas.copy()

# Drop columns not in target schema
tas = tas.drop(columns=['is_aged_care_hub'], errors='ignore')

# Standardise postcode to string
tas['postcode'] = tas['postcode'].astype(str).str.strip().str.split('.').str[0]
tas['postcode'] = tas['postcode'].replace('nan', None)

print(f'TAS rows    : {len(tas)}')
print(f'Null check  :')
print(tas.isnull().sum().to_string())

### 6.3 WA — Map to Target Schema
WA has separate street and postal address columns. We use street address fields.
We also drop irrelevant columns: `_id`, postal address fields, elevation.

In [ ]:
wa = df_wa.copy()

# Map WA columns to target schema using street address fields
wa_clean = pd.DataFrame({
    'name'         : wa['Name'],
    'address'      : wa['Street Address'],
    'suburb'       : wa['Street Suburb'],
    'state'        : wa['Street State'],
    'postcode'     : wa['Street Postcode'].astype(str).str.split('.').str[0],
    'latitude'     : wa['Latitude (Decimal)'],
    'longitude'    : wa['Longitude (Decimal)'],
    'phone'        : wa['Phone'],
    'opening_hours': None,  # not in WA source data
    'venue_type'   : 'Library',
    'source'       : 'Libraries_WA'
})

# Strip whitespace from string columns
str_cols = wa_clean.select_dtypes(include='object').columns
wa_clean[str_cols] = wa_clean[str_cols].apply(lambda col: col.str.strip())

# Drop the 1 row with null address — unusable for location search
before = len(wa_clean)
wa_clean = wa_clean.dropna(subset=['address', 'suburb'])
print(f'Dropped {before - len(wa_clean)} row(s) with null address/suburb')
print(f'WA rows remaining : {len(wa_clean)}')
print(f'Null counts:')
print(wa_clean.isnull().sum().to_string())

### 6.4 SA — Map to Target Schema
SA has `Address1` and `Address2` columns. We combine them where both exist,
otherwise use `Address1` alone as the street address.

In [ ]:
sa = df_sa.copy()

# Combine Address1 and Address2 where Address2 exists
# e.g. 'Level 3, Rundle Place' + '77-91 Rundle Mall' → 'Level 3, Rundle Place, 77-91 Rundle Mall'
sa['full_address'] = sa.apply(
    lambda row: f"{row['Address1']}, {row['Address2']}"
    if pd.notna(row['Address2']) else row['Address1'],
    axis=1
)

# Map to target schema
sa_clean = pd.DataFrame({
    'name'         : sa['Name'],
    'address'      : sa['full_address'],
    'suburb'       : sa['Suburb/town'],
    'state'        : sa['State'],
    'postcode'     : sa['Postcode'].astype(str).str.split('.').str[0],
    'latitude'     : sa['Latitude'],
    'longitude'    : sa['Longitude'],
    'phone'        : sa['Phone'],
    'opening_hours': None,  # not in SA source data
    'venue_type'   : 'Library',
    'source'       : 'Libraries_SA'
})

# Strip whitespace
str_cols = sa_clean.select_dtypes(include='object').columns
sa_clean[str_cols] = sa_clean[str_cols].apply(lambda col: col.str.strip())

# Fill 1 null phone with fallback
sa_clean['phone'] = sa_clean['phone'].fillna('Contact not available')

print(f'SA rows : {len(sa_clean)}')
print(f'Null counts:')
print(sa_clean.isnull().sum().to_string())

---
## 7. Combine All Datasets

Stack all four cleaned datasets into one using `pd.concat`.
This is a union operation — not a join — because each dataset represents different libraries, not the same libraries enriched with extra columns.

In [ ]:
# Select only target schema columns in correct order before stacking
SCHEMA = ['name', 'address', 'suburb', 'state', 'postcode',
          'latitude', 'longitude', 'phone', 'opening_hours',
          'venue_type', 'source']

combined = pd.concat([
    vicqld[SCHEMA],
    tas[SCHEMA],
    wa_clean[SCHEMA],
    sa_clean[SCHEMA]
], ignore_index=True)

print(f'Combined shape : {combined.shape}')
print(f'\nState breakdown:')
print(combined['state'].value_counts().to_string())

---
## 8. Remove Duplicates

Check for duplicate records based on rounded lat/lon (4 decimal places ≈ 11m precision).
If two records are within 11 metres of each other they are almost certainly the same location.

In [ ]:
before = len(combined)

# Round coordinates to 4dp for dedup comparison
combined['lat_r'] = combined['latitude'].round(4)
combined['lon_r'] = combined['longitude'].round(4)

# Keep first occurrence of each coordinate pair
combined = combined.drop_duplicates(subset=['lat_r', 'lon_r'], keep='first')

# Drop helper columns
combined = combined.drop(columns=['lat_r', 'lon_r'])

print(f'Rows before dedup : {before}')
print(f'Rows after dedup  : {len(combined)}')
print(f'Duplicates removed: {before - len(combined)}')

---
## 9. Post-Combination Validation

In [ ]:
print('=== Final Combined Dataset Summary ===')
print(f'Shape          : {combined.shape[0]} rows x {combined.shape[1]} columns')
print(f'\nState coverage :')
print(combined['state'].value_counts().to_string())

In [ ]:
# Null summary
null_pct = (combined.isnull().sum() / len(combined) * 100).round(1)
print('Null Value Summary:')
print(pd.DataFrame({'Null Count': combined.isnull().sum(), 'Null %': null_pct}).to_string())

In [ ]:
# Confirm all coordinates are within Australia's geographic bounds
out_of_bounds = combined[
    (combined['latitude'] < -44) | (combined['latitude'] > -10) |
    (combined['longitude'] < 113) | (combined['longitude'] > 154)
]
print(f'Records outside Australia bounds: {len(out_of_bounds)}')
print(f'Lat range  : {combined["latitude"].min():.4f} to {combined["latitude"].max():.4f}')
print(f'Lon range  : {combined["longitude"].min():.4f} to {combined["longitude"].max():.4f}')

In [ ]:
# Preview final dataset
combined.head(10)

---
## 10. Export Cleaned Combined Dataset

In [ ]:
CLEAN_PATH = os.path.join(clean_path, 'cleaned_Libraries_ALL.csv')

combined = combined.reset_index(drop=True)
combined.to_csv(CLEAN_PATH, index=False, encoding='utf-8')

print(f'Saved to      : {CLEAN_PATH}')
print(f'Final shape   : {combined.shape[0]} rows x {combined.shape[1]} columns')
print(f'\nSource breakdown:')
print(combined['source'].value_counts().to_string())